In [2]:
# Classes and funcctions

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate

from qiskit_aer import AerSimulator
import math

In [3]:
simulator = AerSimulator()

def quantum_random_bit():
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(qc, shots=1).result()
    return int(list(result.get_counts())[0])

def quantum_random_bits(n):
    return [quantum_random_bit() for _ in range(n)]

def encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

def measure_qubit(qc, basis):
    qc = qc.copy()
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(qc, shots=1).result()
    return int(list(result.get_counts())[0])

In [4]:
N_BITS     = 100
CHECK_FRAC = 0.5
THRESHOLD  = 0.11

In [5]:
# === ALICE ===
alice_bits  = quantum_random_bits(N_BITS)
alice_bases = quantum_random_bits(N_BITS)
qubits      = [encode_qubit(b, basis)
               for b, basis in zip(alice_bits, alice_bases)]

print(f"Alice bits  (first 20): {alice_bits[:20]}")
print(f"Alice bases (first 20): {alice_bases[:20]}")

Alice bits  (first 20): [1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1]
Alice bases (first 20): [1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1]


In [ ]:
def eve_intercept(qc):
    """
    Eve's intercept-resend attack on a single qubit.

    Step 1: Eve picks a random basis (she doesn't know Alice's basis)
    Step 2: Eve MEASURES the qubit — this collapses Alice's original
            quantum state. The qubit no longer exists as Alice sent it.
    Step 3: Eve re-encodes her measurement result and forwards a
            brand new qubit to Bob. Bob never receives Alice's original.
    """
    # === EVE ===
    eve_basis  = quantum_random_bit()           # random guess: Z or X
    eve_result = measure_qubit(qc, eve_basis)   
    return encode_qubit(eve_result, eve_basis)  # new qubit sent to Bob

# Eve intercepts EVERY qubit in the channel
qubits_for_bob = [eve_intercept(qc) for qc in qubits]

print(f"Eve has intercepted all {N_BITS} qubits.")
print("Bob will receive Eve's re-encoded qubits, not Alice's originals.")

Eve has intercepted all 100 qubits.
Bob will receive Eve's re-encoded qubits, not Alice's originals.


In [ ]:
# === BOB ===
# Bob receives qubits_for_bob (which he thinks were sent by Alice) actually re-encoded by Eve
bob_bases   = quantum_random_bits(N_BITS)
bob_results = [measure_qubit(qubits_for_bob[i], bob_bases[i])
               for i in range(N_BITS)]

print(f"Bob bases   (first 20): {bob_bases[:20]}")
print(f"Bob results (first 20): {bob_results[:20]}")

Bob bases   (first 20): [0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1]
Bob results (first 20): [0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0]


In [8]:
# === ALICE + BOB (classical channel) ===
matching  = [i for i in range(N_BITS) if alice_bases[i] == bob_bases[i]]
alice_key = [alice_bits[i]  for i in matching]
bob_key   = [bob_results[i] for i in matching]

print(f"Sifted key length: {len(alice_key)} bits")
print(f"Alice sifted (first 20): {alice_key[:20]}")
print(f"Bob   sifted (first 20): {bob_key[:20]}")

Sifted key length: 44 bits
Alice sifted (first 20): [1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0]
Bob   sifted (first 20): [1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1]


In [ ]:
# === ALICE + BOB ===
n_check   = math.ceil(len(alice_key) * CHECK_FRAC)
check_idx = sorted(set(quantum_random_bits(n_check * 3)))[:n_check]

alice_check = [alice_key[i] for i in check_idx]
bob_check   = [bob_key[i]   for i in check_idx]

errors = sum(a != b for a, b in zip(alice_check, bob_check))
qber   = errors / len(check_idx)

final_idx = [i for i in range(len(alice_key)) if i not in check_idx]
final_key = [alice_key[i] for i in final_idx]

print(f"Bits checked : {len(check_idx)}")
print(f"Errors found : {errors}")
print(f"QBER         : {qber:.1%}  (expect ~25% with full intercept)")
print(f"Threshold    : {THRESHOLD:.1%}")
print()

if qber > THRESHOLD:
    print("ATTACK DETECTED! QBER exceeds threshold.")
    print("Key is discarded — communication is aborted.")
else:
    print("No attack detected (Eve bypassed any detection).")
    print(f"Final key ({len(final_key)} bits): {final_key[:30]}...")

Bits checked : 2
Errors found : 1
QBER         : 50.0%  (expect ~25% with full intercept)
Threshold    : 11.0%

ATTACK DETECTED! QBER exceeds threshold.
Key is discarded — communication is aborted.


In [ ]:
# Eve only intercepts a fraction of qubits to try to stay under the threshold.
# This shows WHY Alice and Bob need a large enough sample.

EVE_FRACTION = 0.3   # Eve intercepts only 30% of qubits

qubits_partial = []
for i, qc in enumerate(qubits):
    if quantum_random_bit() == 0 and quantum_random_bit() == 0:
        # ~25% chance of intercepting (two 0s in a row)
        qubits_partial.append(eve_intercept(qc))
    else:
        qubits_partial.append(qc)   # original qubit passes through

bob_results_partial = [measure_qubit(qubits_partial[i], bob_bases[i])
                       for i in range(N_BITS)]

matching_p  = [i for i in range(N_BITS) if alice_bases[i] == bob_bases[i]]
alice_key_p = [alice_bits[i]        for i in matching_p]
bob_key_p   = [bob_results_partial[i] for i in matching_p]

errors_p = sum(a != b for a, b in zip(alice_key_p, bob_key_p))
qber_p   = errors_p / len(alice_key_p)

print(f"Partial intercept QBER: {qber_p:.1%}")
print("Eve intercepts less → lower QBER → harder to detect,but she also learns less of the key.")

Partial intercept QBER: 2.3%
Eve intercepts less → lower QBER → harder to detect,
but she also learns less of the key. There is no free lunch for Eve.
